# This notebook create an agentic task that using crewai

the crew will take a Jira ticket that describes a change needed to be done in a file
and will implement it, the change will be set to have a pull request
and the pull request 

The Tasks are to do that are:
1. TicketHandler - A task that recives the ticket
2. TechnicalWritter - A task perform the change in file and commit
3. SlackAnnouncer - A task perform a post to slack channel

The Agents are:
1. Jira - Using Task TicketHandler  
2. GitHub Writer - Using Task TechnicalWritter
3. Announcer - Using Task SlackAnnouncer

The Tools: 
1. Github - Will be used by GitHubWriter Agent
2. Slack - Will be used by Announcer Agent


In [1]:
import requests
import json
from src.server.controllers.jira_controller import JiraController
from src.server.models.jira_config import JiraConfig
from src.server.tools.jira_service import JiraService

# Base URL for the Cerebrum API
base_url = "http://localhost:8080"

# Health check
response = requests.get(f"{base_url}/health_check")
if response.status_code == 200:
    print("Cerebrum API is running and healthy.")
else:
    print(f"Error: Cerebrum API is not responding. Status code: {response.status_code}")
    raise SystemExit("Cannot proceed without a healthy Cerebrum API.")


All OK: "All OK"


In [15]:
# DevSecOps CrewAI Configuration
crew_config = {
    "description": "A crew for automating DevOps tasks in a microservices architecture",
    "agents": [
        {
            "role": "Jira Analyst",
            "goal": "Analyze Jira tickets and extract relevant information for DevOps tasks",
            "backstory": "You are an experienced Jira analyst with a deep understanding of DevOps practices and microservices architecture.",
            "tools": [
                {"name": "jira_tool", "description": "Tool for interacting with Jira"}
            ]
        },
        {
            "role": "DevOps Engineer",
            "goal": "Implement necessary changes in Terraform configurations and create GitHub branches",
            "backstory": "You are a skilled DevOps engineer with expertise in Terraform, Kubernetes, and microservices deployment.",
            "tools": [
                {"name": "github_tool", "description": "Tool for interacting with GitHub"},
                {"name": "terraform_tool", "description": "Tool for modifying Terraform configurations"}
            ]
        },
        {
            "role": "Code Reviewer",
            "goal": "Review changes, create pull requests, and ensure code quality",
            "backstory": "You are a meticulous code reviewer with a strong background in DevOps practices and security considerations.",
            "tools": [
                {"name": "github_tool", "description": "Tool for interacting with GitHub"}
            ]
        }
    ],
    "tasks": [
        {
            "description": "Analyze the Jira ticket and extract information about the required microservice communication changes",
            "expected_output": "A JSON object containing the source service, destination service, and communication type (ingress/egress)",
            "agent_role": "Jira Analyst"
        },
        {
            "description": "Create a new GitHub branch for the changes and update the Terraform configurations",
            "expected_output": "A JSON object containing the branch name and a list of modified Terraform files",
            "agent_role": "DevOps Engineer"
        },
        {
            "description": "Review the changes, create a pull request, and update the Jira ticket with the PR link",
            "expected_output": "A JSON object containing the pull request URL and updated Jira ticket status",
            "agent_role": "Code Reviewer"
        }
    ],
    "tools_proxy_collection": {
        "toolsets": [
            {"id": "devops_toolset", "name": "DevOpsTools"}
        ],
        "functions": [
            {"name": "terraform_tool", "type": "function"}
        ],
        "objects": [
            {"name": "github_tool", "object_type": "github_tool"},
            {"name": "jira_tool", "object_type": "jira_tool"}
        ]
    }
}

# Create tasksmanager
tasksmanager_data = {
    "name": "DevSecOps Automation Manager",
    "model": "gpt-4",
    "description": "TasksManager for DevSecOps automation tasks",
}
tasksmanager_response = requests.post(f"{base_url}/tasksmanager", json=tasksmanager_data)
assert tasksmanager_response.status_code == 201, f"Failed to create tasksmanager: {tasksmanager_response.text}"
tasksmanager_id = tasksmanager_response.json()["id"]
print(f"TasksManager created with ID: {tasksmanager_id}")

# Create CrewAI configuration
crew_response = requests.post(f"{base_url}/agent/crewai", json=crew_config)
assert crew_response.status_code == 200, f"Failed to create CrewAI configuration: {crew_response.text}"
crew_data = crew_response.json()
crew_id = crew_data['crew_id']['id']
curried_proxy_id = crew_data['curried_proxy_id']
print(f"CrewAI configuration created with ID: {crew_id}")

# Create task
task_data = {
    "name": "DevSecOps Automation",
    "description": "Automate DevOps tasks for microservices communication",
    "tasksmanager_id": tasksmanager_id,
    "crewai_info_id": crew_id,
    "curried_proxy_id": curried_proxy_id,
}
task_response = requests.post(f"{base_url}/task", json=task_data)
assert task_response.status_code == 201, f"Failed to create task: {task_response.text}"
task_id = task_response.json()["id"]
print(f"Task created with ID: {task_id}")

201


In [ ]:
import requests

crew_config = {
    "description": "File creator",
    "agents": [
        {
            "role": "File maker",
            "verbose": True,
            "goal": "create new files and save them",
            "backstory": "You are a 40 year old administrator that creates files and stores them",
            "class_tools": [],
            "tools": [
                {"name": "save_file", "description": "Tool for storing files"},
                {"name": "GitHubTool", "description": "Tool for storing files"},
            ],
        }
    ],
    "tasks": [
        {
            "description": "Make files",
            "expected_output": "Create a .txt file that says hello world and write it to /home/egbert/hello_world.txt",
            "agent_role": "File maker"
        },
        {
            "description": "change and commit file to github repo",
            "expected_output": "Change the file hello.txt to say hello world and commit it to the github repo GenBrainAI/cerebrum",
            "agent_role": "File maker file"
        },
        {
            "description": "get the readme from the repository and output its contents",
            "expected_output": "Get the readme file from the github repo and output its contents",
            "agent_role": "File maker"
        }
    ]
}
base_url = "http://localhost:8080"

response = requests.post(f"{base_url}/agent/crewai", json=crew_config)
assert response.status_code == 200
data = response.json()
print(data)

In [ ]:
# Jira configuration
jira_config = JiraConfig(
    project_key="DEVOPS",  # Replace with your actual Jira project key
    jira_url="https://your-jira-instance.atlassian.net",  # Replace with your Jira URL
    jira_username="your-jira-username",  # Replace with your Jira username
    jira_api_token="your-jira-api-token",  # Replace with your Jira API token
    task_id=task_id,
    schedule_interval=5  # Check every 5 minutes
)

# Create JiraService
jira_service = JiraService(jira_config.jira_url, jira_config.jira_username, jira_config.jira_api_token)

# Create JiraController
jira_controller = JiraController(jira_config, jira_service)

print("Jira Controller set up successfully.")

In [ ]:
# Prepare the state data
state_data = {
    "tasksmanager_id": tasksmanager_id,
    "crew_id": crew_id,
    "task_id": task_id,
    "jira_config": jira_config.dict(),
}

# Save the state
save_state_url = f"{base_url}/state/save"
save_state_data = {
    "type": "file",  # You can change this to 's3' or 'postgres' if needed
    "location": "./",  # Change this to the desired save location
    "state_name": "devsecops_state.json"
}

save_response = requests.post(save_state_url, json=save_state_data)
assert save_response.status_code == 201, f"Failed to save state: {save_response.text}"
print("State saved successfully.")

In [ ]:
# Load the state
load_state_url = f"{base_url}/state/load"
load_state_data = {
    "type": "file",  # Make sure this matches the type used in saving
    "location": "./",  # Make sure this matches the location used in saving
    "state_name": "devsecops_state.json"
}

load_response = requests.post(load_state_url, json=load_state_data)
assert load_response.status_code == 200, f"Failed to load state: {load_response.text}"
loaded_state = load_response.json()

# Extract the loaded data
tasksmanager_id = loaded_state["tasksmanager_id"]
crew_id = loaded_state["crew_id"]
task_id = loaded_state["task_id"]
jira_config = JiraConfig(**loaded_state["jira_config"])

print("State loaded successfully.")
print(f"Loaded TasksManager ID: {tasksmanager_id}")
print(f"Loaded Crew ID: {crew_id}")
print(f"Loaded Task ID: {task_id}")